# Validate LLM — Court InfoCuria Commission References v5

Two-pass validation of the Commission-only outputs produced by
`run_regex_court_infocuria_v9`:

1. Commission internal case numbers (`COMP`, `IV`, `AT`)
2. Published Commission decision references (`YYYY/NNN/EEC`, `EC`, `ECSC`,
   `EU`, and multilingual equivalents)

## v5 purpose

This version is configured primarily to **rebuild the Excel outputs from the
existing v4 JSONL checkpoint without rerunning the LLM validation**.

Default:

```python
RUN_MODE = "rebuild_excel"
```

The existing checkpoint remains:

```text
court_infocuria_commission_two_prompts_validated_v4.jsonl
```

The rebuilt Excel files receive a new `v5` basename. The notebook does not
rename, overwrite, or append to the v4 JSONL while operating in
`rebuild_excel` mode.

All paths, labels, source types, and output names use **InfoCuria** terminology.


In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, Iterable, List
from datetime import datetime
import asyncio
import hashlib
import json
import re
import time

import pandas as pd

try:
    import requests
    REQUESTS_AVAILABLE = True
except Exception:
    REQUESTS_AVAILABLE = False

try:
    import aiohttp
    AIOHTTP_AVAILABLE = True
except Exception:
    AIOHTTP_AVAILABLE = False

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **kwargs: x

NOTEBOOK_DIR = Path.cwd().resolve()


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (
            (candidate / "output").exists()
            or (candidate / "code").exists()
            or (candidate / "data").exists()
        ):
            return candidate
    return start


PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
OUTPUT_DIR = PROJECT_ROOT / "output" / "court_infocuria" / "excel"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Deduplicated InfoCuria regex inputs.
input_caseno_path = (
    OUTPUT_DIR / "court_infocuria_commission_caseno_hit_deduplicated.xlsx"
)
input_ref_path = (
    OUTPUT_DIR / "court_infocuria_commission_reference_hit_deduplicated.xlsx"
)

# Optional upload fallbacks for running outside the project tree.
UPLOAD_FALLBACK_CASENO_PATHS = [
    Path("/mnt/data/court_infocuria_commission_caseno_hit_deduplicated.xlsx")
]
UPLOAD_FALLBACK_REF_PATHS = [
    Path("/mnt/data/court_infocuria_commission_reference_hit_deduplicated.xlsx")
]

# The completed v4 checkpoint is the source of truth for rebuilding.
CHECKPOINT_BASENAME = "court_infocuria_commission_two_prompts_validated_v4"
PROJECT_CHECKPOINT_JSONL_PATH = OUTPUT_DIR / f"{CHECKPOINT_BASENAME}.jsonl"
UPLOAD_FALLBACK_JSONL_PATHS = [
    Path("/mnt/data/court_infocuria_commission_two_prompts_validated_v4.jsonl")
]

# v5 output names. The v4 checkpoint remains unchanged.
OUTPUT_BASENAME = "court_infocuria_commission_two_prompts_validated"

output_caseno_path = (
    OUTPUT_DIR / "court_infocuria_commission_caseno_hit_validated.xlsx"
)
output_ref_path = (
    OUTPUT_DIR / "court_infocuria_commission_reference_hit_validated.xlsx"
)
COMBINED_CLASSIFIED_EXCEL_PATH = OUTPUT_DIR / f"{OUTPUT_BASENAME}_combined.xlsx"
DOCUMENT_SUMMARY_PATH = OUTPUT_DIR / f"{OUTPUT_BASENAME}_document_summary.xlsx"
RERUN_CANDIDATES_PATH = OUTPUT_DIR / f"{OUTPUT_BASENAME}_rerun_candidates.xlsx"

# Safe default: rebuild Excel only. No model requests are sent.
RUN_MODE = "rebuild_excel"  # "rebuild_excel" | "update" | "redo_all"
MAX_ROWS_PER_INPUT = None
DRY_RUN = False

# These settings matter only if RUN_MODE is deliberately changed.
API_URL = "http://localhost:8000/v1/chat/completions"
MODEL = "google/gemma-4-26B-A4B-it"
TEMPERATURE = 0.0
MAX_TOKENS = 800
REQUEST_TIMEOUT = 240

ASYNC_CONCURRENT_REQUESTS = True
CONCURRENT_REQUESTS = 8
RETRY_ERRORS_SEQUENTIALLY = True
RETRY_MAX_ROWS = None
USE_ASYNC_PROGRESS_BAR = True

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Case-number input:", input_caseno_path)
print("Decision-reference input:", input_ref_path)
print("v4 checkpoint:", PROJECT_CHECKPOINT_JSONL_PATH)
print("v5 combined Excel:", COMBINED_CLASSIFIED_EXCEL_PATH)
print("RUN_MODE:", RUN_MODE)

if RUN_MODE == "rebuild_excel":
    print("SAFE MODE: existing JSONL will be read; no LLM calls will be made.")
else:
    print(
        "WARNING: RUN_MODE is not 'rebuild_excel'. "
        "The notebook may send LLM requests."
    )


PROJECT_ROOT: /home/edik/projects/eccjeu
OUTPUT_DIR: /home/edik/projects/eccjeu/output/court_infocuria/excel
Case-number input: /home/edik/projects/eccjeu/output/court_infocuria/excel/court_infocuria_commission_caseno_hit_deduplicated.xlsx
Decision-reference input: /home/edik/projects/eccjeu/output/court_infocuria/excel/court_infocuria_commission_reference_hit_deduplicated.xlsx
v4 checkpoint: /home/edik/projects/eccjeu/output/court_infocuria/excel/court_infocuria_commission_two_prompts_validated_v4.jsonl
v5 combined Excel: /home/edik/projects/eccjeu/output/court_infocuria/excel/court_infocuria_commission_two_prompts_validated_v5_combined.xlsx
RUN_MODE: rebuild_excel
SAFE MODE: existing JSONL will be read; no LLM calls will be made.


## 1. Load the two Commission-only deduplicated InfoCuria regex outputs

These are the two workbooks written by `run_regex_court_infocuria_v9`.


In [2]:
def resolve_input_path(project_path: Path, fallback_paths: Iterable[Path]) -> Path:
    candidates = [project_path, *list(fallback_paths)]
    for path in candidates:
        if path.exists():
            return path
    tried = " | ".join(str(path) for path in candidates)
    raise FileNotFoundError(f"Could not find input workbook. Tried: {tried}")


def read_input(project_path: Path, fallback_paths: Iterable[Path], source_dataset: str) -> pd.DataFrame:
    path = resolve_input_path(project_path, fallback_paths)
    df = pd.read_excel(path)
    df.columns = [str(c).strip() for c in df.columns]
    df["source_dataset"] = source_dataset
    df["source_input_file"] = path.name
    # These two upstream files are Commission-hit-only views and do not carry
    # the older generic `case_system` column.
    if "case_system" not in df.columns:
        df["case_system"] = "commission"
    print(f"Loaded {len(df):,} rows and {len(df.columns):,} columns from {path}")
    return df


caseno_cases = read_input(input_caseno_path, UPLOAD_FALLBACK_CASENO_PATHS, "caseno")
reference_cases = read_input(input_ref_path, UPLOAD_FALLBACK_REF_PATHS, "reference")


def resolve_checkpoint_path(
    project_path: Path,
    fallback_paths: Iterable[Path],
) -> Path:
    candidates = [project_path, *list(fallback_paths)]
    for path in candidates:
        if path.exists():
            return path
    tried = " | ".join(str(path) for path in candidates)
    raise FileNotFoundError(
        "Could not find the completed InfoCuria JSONL checkpoint. "
        f"Tried: {tried}"
    )


CHECKPOINT_JSONL_PATH = resolve_checkpoint_path(
    PROJECT_CHECKPOINT_JSONL_PATH,
    UPLOAD_FALLBACK_JSONL_PATHS,
)
print("Resolved JSONL checkpoint:", CHECKPOINT_JSONL_PATH)

# The downstream prompt expects these canonical fields. Accept a few reasonable
# aliases if an upstream workbook used family-specific column names.
def harmonize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    aliases = {
        "normalized_case_number": [
            "normalized_commission_case_number",
            "normalized_commission_decision_reference",
            "cleaned_case_number",
            "normalized_reference",
        ],
        "normalized_candidate": ["proposed_normalized_candidate", "cleaned_reference"],
        "matched_texts": ["matched_text", "raw_match", "reference_text"],
        "document_case_key": ["document_identifier_key", "document_reference_key", "dedup_key"],
        "commission_reference_type": ["commission_reference_types", "reference_category"],
    }
    for target, candidates in aliases.items():
        if target not in df.columns:
            for candidate in candidates:
                if candidate in df.columns:
                    df[target] = df[candidate]
                    break
    if "normalized_candidate" not in df.columns:
        df["normalized_candidate"] = df.get("normalized_case_number", "")
    return df

caseno_cases = harmonize_columns(caseno_cases)
reference_cases = harmonize_columns(reference_cases)

required_columns = [
    "document_id",
    "case_system",
    "normalized_case_number",
    "matched_texts",
]
for label, df in [("caseno", caseno_cases), ("reference", reference_cases)]:
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"{label} input is missing required columns: {missing}")

    non_commission = ~df["case_system"].fillna("").astype(str).str.lower().eq("commission")
    if non_commission.any():
        raise ValueError(
            f"{label} input contains {int(non_commission.sum()):,} non-Commission rows. "
            "These validators must only receive Commission-hit rows."
        )

cases = pd.concat([caseno_cases, reference_cases], ignore_index=True, sort=False)
print(f"Combined rows available for validation: {len(cases):,}")
cases.groupby("source_dataset", dropna=False).size()


Loaded 4,936 rows and 51 columns from /home/edik/projects/eccjeu/output/court_infocuria/excel/court_infocuria_commission_caseno_hit_deduplicated.xlsx
Loaded 10,047 rows and 51 columns from /home/edik/projects/eccjeu/output/court_infocuria/excel/court_infocuria_commission_reference_hit_deduplicated.xlsx
Resolved JSONL checkpoint: /home/edik/projects/eccjeu/output/court_infocuria/excel/court_infocuria_commission_two_prompts_validated_v4.jsonl
Combined rows available for validation: 14,983


source_dataset
caseno        4936
reference    10047
dtype: int64

## 2. Stable IDs and row selection

In [3]:
def clean_scalar(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()

def stable_hash(parts: Iterable[Any], n: int = 16) -> str:
    raw = "||".join(clean_scalar(p) for p in parts)
    return hashlib.sha1(raw.encode("utf-8", errors="replace")).hexdigest()[:n]

def make_llm_case_id(row: pd.Series) -> str:
    dataset = clean_scalar(row.get("source_dataset")) or "unknown"
    existing = clean_scalar(row.get("document_case_key"))
    if existing:
        return f"{dataset}::{existing}"
    return f"{dataset}::{stable_hash([
        row.get('document_id'),
        row.get('case_system'),
        row.get('normalized_case_number'),
        row.get('matched_texts'),
    ])}"

cases = cases.copy()
cases["llm_case_id"] = cases.apply(make_llm_case_id, axis=1)
cases["source_type"] = cases["source_dataset"].map({"caseno": "court_infocuria_commission_caseno_hit", "reference": "court_infocuria_commission_reference_hit"})

work_cases = cases.copy()
if MAX_ROWS_PER_INPUT is not None:
    work_cases = (
        work_cases.groupby("source_dataset", group_keys=False, sort=False)
        .head(int(MAX_ROWS_PER_INPUT))
        .copy()
    )

print(f"Rows selected: {len(work_cases):,} / {len(cases):,}")
print(work_cases.groupby("source_dataset").size().to_string())
work_cases[[
    c for c in [
        "llm_case_id", "document_id", "normalized_case_number",
        "matched_texts", "n_hits", "paragraph_text_agg"
    ] if c in work_cases.columns
]].head(10)


Rows selected: 14,983 / 14,983
source_dataset
caseno        4936
reference    10047


,llm_case_id,document_id,normalized_case_number,matched_texts,n_hits,paragraph_text_agg
0,caseno::1055241||commission_case_number||COMP/...,1055241,COMP/C2/38.698,COMP/C2/38.698,1,BESCHIKKING VAN HET HOF (Achtste kamer) 14 jan...
1,caseno::1204995||commission_case_number||COMP/...,1204995,COMP/E-1/36.212,COMP/E‐1/36.212,2,JUDGMENT OF THE COURT (Third Chamber) 3 Septem...
2,caseno::1211704||commission_case_number||COMP/...,1211704,COMP/38.543,COMP/38.543,2,JUDGMENT OF THE GENERAL COURT (Eighth Chamber)...
3,caseno::1211705||commission_case_number||COMP/...,1211705,COMP/F/38.620,COMP/F/38.620,2,"JUDGMENT OF THE GENERAL COURT (Sixth Chamber, ..."
4,caseno::1211705||commission_case_number||COMP/...,1211705,COMP/F/38.645,COMP/F/38.645,1,"JUDGMENT OF THE GENERAL COURT (Sixth Chamber, ..."
5,caseno::1211708||commission_case_number||COMP/...,1211708,COMP/38.543,COMP/38.543,3,JUDGMENT OF THE GENERAL COURT (Eighth Chamber)...
6,caseno::1211708||commission_case_number||COMP/...,1211708,COMP/F-1/38.121,COMP/F-1/38.121,1,JUDGMENT OF THE GENERAL COURT (Eighth Chamber)...
7,caseno::1211709||commission_case_number||COMP/...,1211709,COMP/F/38.620,COMP/F/38.620,2,"JUDGMENT OF THE GENERAL COURT (Sixth Chamber, ..."
8,caseno::1211709||commission_case_number||IV/32...,1211709,IV/32.448,IV/32.448,2,"JUDGMENT OF THE GENERAL COURT (Sixth Chamber, ..."
9,caseno::1211709||commission_case_number||IV/32...,1211709,IV/32.450,IV/32.450,2,"JUDGMENT OF THE GENERAL COURT (Sixth Chamber, ..."


## 3. Commission-focused prompt and output schema

In [4]:
ALLOWED_VALIDATION_STATUS = [
    "valid_commission_reference",
    "malformed_but_recoverable",
    "false_positive",
    "unknown_unclear",
]
ALLOWED_CONFIDENCE = ["high", "medium", "low"]

def prompt_scalar(value: Any) -> str:
    """
    Preserve prompt evidence verbatim.

    Unlike clean_scalar(), this does not collapse whitespace, line breaks,
    punctuation, or OCR artefacts. Those details may be decisive.
    """
    if value is None:
        return ""
    if isinstance(value, float) and pd.isna(value):
        return ""
    return str(value)

def get_paragraph_for_prompt(row: pd.Series) -> str:
    for col in ["paragraph_text_agg", "paragraph_text", "paragraph"]:
        value = prompt_scalar(row.get(col))
        if value:
            return value
    return ""

def get_context_for_prompt(row: pd.Series) -> str:
    for col in ["context_180_agg", "context_150_agg", "context_agg", "context"]:
        value = prompt_scalar(row.get(col))
        if value:
            return value
    return ""

CASENO_SYSTEM_PROMPT = r"""
You validate regex candidates extracted from EU Court documents.

This pass validates exactly one reference family:
EUROPEAN COMMISSION CASE NUMBERS.

A valid case number:
- begins with the uppercase prefix COMP, IV, or AT;
- is followed by a structured sequence of letters and digits separated by
  slashes, dots, hyphens, dashes, or OCR-damaged spaces;
- functions as an actual European Commission case/file number in context.

Examples of valid or recoverable structures:
- IV/35.860
- IV/A-00004-03344
- COMP/E-1/38.113
- COMP/F-3/E 2/B1/00004/00093
- AT.39740
- OCR-spaced or punctuation-damaged versions of the same kinds of identifiers.

The numeric core usually contains groups of approximately 2 to 5 digits,
although exceptional administrative subcomponents may contain leading zeroes
or longer structured segments.

The prefix must genuinely be COMP, IV, or AT as an identifier prefix.
Do not accept:
- the word COMPETITION or another ordinary word merely beginning with COMP;
- lowercase ordinary prose such as "at";
- Court case numbers;
- dates, article numbers, paragraph numbers, pages, amounts, or unrelated codes;
- Commission decision references such as 82/861/EEC;
- M, SA, C(...), K(...), or other Commission identifier families in this task.

A malformed candidate is recoverable only when the supplied raw match,
proposed normalization, paragraph, and context strongly support one specific
COMP/IV/AT case number. Do not invent digits, letters, separators, or prefixes.

MALFORMATION STANDARD

Use malformed_but_recoverable only when:
1. the raw candidate is genuinely damaged by OCR, spacing, punctuation, or
   regex spillover;
2. the surrounding evidence supports one clear intended identifier; and
3. the normalized form can be reconstructed without speculation.

If more than one reconstruction is plausible, use unknown_unclear.
If the token is clearly something else, use false_positive.

EVIDENCE PRIORITY

Use all supplied evidence without shortening:
1. raw matched text;
2. proposed normalized candidate;
3. full paragraph;
4. local context window.

The paragraph/context decide meaning. The normalized candidate is only a
proposal and may be wrong.

Return strict JSON only.
""".strip()

REFERENCE_SYSTEM_PROMPT = r"""
You validate regex candidates extracted from EU Court documents.

This pass validates exactly one reference family:
EUROPEAN COMMISSION DECISION REFERENCES.

Typical structures are:
- 80/234
- 82/861/EEC
- 94/815/EC
- 2003/382/EC

The first numeric group is normally exactly 2 or 4 digits.
The second group is normally 1 to 4 digits.
A final separator and Community/Union suffix may be present, but is optional,
especially for older decisions.

Common suffix families include:
- EEC: EEC, EWG, CEE, EEG, EØF, ETY, EGK, ΕΟΚ, ЕИО
- EC: EC, EG, CE, EF, EY, EK, KE, ΕΚ, ЕО
- ECSC: ECSC, EGKS, CECA, EKSF, EHTY, ΕΚΑΧ, ЕОВС
- EU: EU, UE, ΕΕ, ЕС

OCR may:
- remove spaces: Commission Decision94/815/EC;
- insert spaces around or inside separators;
- replace a slash with a hyphen, dash, or occasionally a dot;
- attach following prose to a suffix;
- split or slightly damage the number.

Institutional context is mandatory:
- the nearby text must identify the act as a Commission decision;
- equivalents of both "Commission" and "decision" may appear in any EU language;
- their order does not matter;
- wording such as "décision 82/861 de la Commission" is valid;
- the reference may lack a suffix when the Commission-decision context is clear.

Reject:
- Council decisions, even if the word Commission appears elsewhere nearby;
- Parliament decisions;
- Court judgments, orders, or Court case numbers;
- Commission case numbers beginning COMP, IV, or AT;
- directives, regulations, recommendations, framework decisions, or other
  instruments that are not Commission decisions;
- a bare number such as 80/85 when the context does not clearly attach it to a
  Commission decision.

An ambiguous 2-digit/2-digit form may be accepted when the immediate wording
clearly identifies it as a Commission decision. It must be rejected when it is
clearly a Court case number or another type of reference.

MALFORMATION STANDARD

Use malformed_but_recoverable only when:
1. the raw candidate is genuinely damaged by OCR, spacing, punctuation, or
   regex spillover;
2. the surrounding evidence supports one clear intended identifier; and
3. the normalized form can be reconstructed without speculation.

If more than one reconstruction is plausible, use unknown_unclear.
If the token is clearly something else, use false_positive.

EVIDENCE PRIORITY

Use all supplied evidence without shortening:
1. raw matched text;
2. proposed normalized candidate;
3. full paragraph;
4. local context window.

The paragraph/context decide institutional meaning. The normalized candidate
is only a proposal and may be wrong.

Return strict JSON only.
""".strip()

USER_PROMPT_CASENO_TEMPLATE = r"""
Validate the supplied candidate as a European Commission case number. Do not
classify the Court document as a whole and do not validate decision references.

DOCUMENT METADATA
document_id={document_id}
court_case_number={court_case_number}
title={document_title}
date={decision_date}

CANDIDATE DATA
document_case_key={document_case_key}
case_system_precheck={case_system}
raw_matched_text:
<<<MATCHED_TEXT
{matched_texts}
MATCHED_TEXT

proposed_normalized_case_number:
<<<NORMALIZED_CASE_NUMBER
{normalized_case_number}
NORMALIZED_CASE_NUMBER

proposed_normalized_candidate:
<<<NORMALIZED_CANDIDATE
{normalized_candidate}
NORMALIZED_CANDIDATE

regex_reference_type={commission_reference_type}
regex_hit_types={hit_types}
regex_commission_units={commission_units}
n_hits={n_hits}

FULL PARAGRAPH — VERBATIM, NOT SHORTENED
<<<PARAGRAPH
{paragraph}
PARAGRAPH

LOCAL CONTEXT WINDOW — VERBATIM, NOT SHORTENED
<<<CONTEXT
{context}
CONTEXT

DECISION PROCESS

- The intended identifier must begin with uppercase COMP, IV, or AT.
- Verify that the prefix is an identifier, not part of an ordinary word.
- Verify that the remaining letters/digits and separators form a plausible
  Commission case-number structure.
- OCR spacing and punctuation damage are allowed only when one defensible
  normalization is clear from the evidence.
- Reject decision references, C(YYYY) N, K(YYYY) N, M, SA, Court case numbers,
  and generic administrative numbers.

Then classify as valid_commission_reference, malformed_but_recoverable,
false_positive, or unknown_unclear.

OUTPUT RULES

- reference_type must be commission_case_number, none, or unknown.
- cleaned_reference must contain only the corrected candidate.
- Preserve meaningful uppercase prefixes.
- Do not silently add missing digits or letters.
- If false_positive, use reference_type="none", cleaned_reference="", and an
  empty all_commission_references list.
- all_commission_references may include other genuine COMP/IV/AT case numbers
  visible in the supplied paragraph/context, but must not include decision
  references or other excluded families.
- evidence should reproduce the shortest decisive wording while preserving
  enough text to show why the candidate is or is not valid.
- normalization_notes should state exactly what was repaired, or be empty for
  a clean reference.
- reason must explain syntax and why the candidate is or is not a Commission
  case number.

Return exactly these JSON keys:
{{
  "is_valid_commission_reference": true,
  "reference_type": "commission_case_number | none | unknown",
  "validation_status": "valid_commission_reference | malformed_but_recoverable | false_positive | unknown_unclear",
  "cleaned_reference": "",
  "all_commission_references": [],
  "additional_commission_case_text": "",
  "evidence": "",
  "normalization_notes": "",
  "confidence": "high | medium | low",
  "reason": ""
}}
""".strip()

USER_PROMPT_REFERENCE_TEMPLATE = r"""
Validate the supplied candidate as a European Commission decision reference.
Do not classify the Court document as a whole and do not validate COMP/IV/AT
case numbers.

DOCUMENT METADATA
document_id={document_id}
court_case_number={court_case_number}
title={document_title}
date={decision_date}

CANDIDATE DATA
document_case_key={document_case_key}
case_system_precheck={case_system}
raw_matched_text:
<<<MATCHED_TEXT
{matched_texts}
MATCHED_TEXT

proposed_normalized_case_number:
<<<NORMALIZED_CASE_NUMBER
{normalized_case_number}
NORMALIZED_CASE_NUMBER

proposed_normalized_candidate:
<<<NORMALIZED_CANDIDATE
{normalized_candidate}
NORMALIZED_CANDIDATE

regex_reference_type={commission_reference_type}
regex_hit_types={hit_types}
regex_commission_units={commission_units}
regex_decision_suffixes={decision_suffixes}
n_hits={n_hits}

FULL PARAGRAPH — VERBATIM, NOT SHORTENED
<<<PARAGRAPH
{paragraph}
PARAGRAPH

LOCAL CONTEXT WINDOW — VERBATIM, NOT SHORTENED
<<<CONTEXT
{context}
CONTEXT

DECISION PROCESS

- The intended number normally has exactly 2 or 4 digits, a separator, and
  1 to 4 digits.
- A final separator plus a recognized multilingual EC/EEC/ECSC/EU suffix is
  strong evidence but is not mandatory.
- Both a word meaning Commission and a word meaning decision should occur in
  the nearby paragraph/context, in any order and any EU language.
- Confirm that those words actually identify this candidate as a Commission
  decision.
- Reject a Council decision or another institution's decision even if the
  Commission is mentioned elsewhere.
- Reject COMP/IV/AT case numbers.
- Treat a short form such as 80/85 as plausible only when the immediate
  Commission-decision wording makes that interpretation convincing.

Then classify as valid_commission_reference, malformed_but_recoverable,
false_positive, or unknown_unclear.

OUTPUT RULES

- reference_type must be commission_decision_reference, none, or unknown.
- cleaned_reference must contain only the corrected candidate.
- Preserve meaningful uppercase suffixes.
- Do not silently add missing digits or letters.
- If false_positive, use reference_type="none", cleaned_reference="", and an
  empty all_commission_references list.
- all_commission_references may include other genuine Commission decision
  references visible in the supplied paragraph/context, but must not include
  COMP/IV/AT case numbers or other excluded families.
- evidence should reproduce the shortest decisive wording while preserving
  enough text to show why the candidate is or is not valid.
- normalization_notes should state exactly what was repaired, or be empty for
  a clean reference.
- reason must explain syntax, institutional authorship, and why the candidate
  is or is not a Commission decision reference.

Return exactly these JSON keys:
{{
  "is_valid_commission_reference": true,
  "reference_type": "commission_decision_reference | none | unknown",
  "validation_status": "valid_commission_reference | malformed_but_recoverable | false_positive | unknown_unclear",
  "cleaned_reference": "",
  "all_commission_references": [],
  "additional_commission_case_text": "",
  "evidence": "",
  "normalization_notes": "",
  "confidence": "high | medium | low",
  "reason": ""
}}
""".strip()

PROMPT_FIELDS = {
    "document_id": lambda row: prompt_scalar(row.get("document_id")),
    "court_case_number": lambda row: prompt_scalar(row.get("case_number_clean", row.get("case_number"))),
    "document_title": lambda row: prompt_scalar(row.get("document_title", row.get("title"))),
    "decision_date": lambda row: prompt_scalar(row.get("decision_date", row.get("date"))),
    "document_case_key": lambda row: prompt_scalar(row.get("document_case_key")),
    "case_system": lambda row: prompt_scalar(row.get("case_system")),
    "matched_texts": lambda row: prompt_scalar(row.get("matched_texts")),
    "normalized_case_number": lambda row: prompt_scalar(row.get("normalized_case_number")),
    "normalized_candidate": lambda row: prompt_scalar(row.get("normalized_candidate")),
    "commission_reference_type": lambda row: prompt_scalar(row.get("commission_reference_type")),
    "hit_types": lambda row: prompt_scalar(row.get("hit_types")),
    "commission_units": lambda row: prompt_scalar(row.get("commission_units")),
    "decision_suffixes": lambda row: prompt_scalar(row.get("decision_suffixes")),
    "n_hits": lambda row: prompt_scalar(row.get("n_hits")),
    "paragraph": get_paragraph_for_prompt,
    "context": get_context_for_prompt,
}

def get_prompt_family(row: pd.Series) -> str:
    source_dataset = clean_scalar(row.get("source_dataset")).lower()
    if source_dataset == "caseno":
        return "caseno"
    if source_dataset == "reference":
        return "reference"
    raise ValueError(f"Unknown source_dataset for prompt routing: {source_dataset!r}")

def get_system_prompt(row: pd.Series) -> str:
    return CASENO_SYSTEM_PROMPT if get_prompt_family(row) == "caseno" else REFERENCE_SYSTEM_PROMPT

def build_prompt(row: pd.Series) -> str:
    values = {name: getter(row) for name, getter in PROMPT_FIELDS.items()}
    template = USER_PROMPT_CASENO_TEMPLATE if get_prompt_family(row) == "caseno" else USER_PROMPT_REFERENCE_TEMPLATE
    return template.format(**values)



## 4. JSON parsing and LLM helpers

In [5]:
def _strip_fences(text: str) -> str:
    text = (text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return text.strip()

def extract_json_object(text: str) -> Dict[str, Any]:
    text = _strip_fences(text)
    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            if not obj:
                raise ValueError("Empty JSON list")
            obj = obj[0]
        if not isinstance(obj, dict):
            raise ValueError("Response is not a JSON object")
        return obj
    except Exception:
        match = re.search(r"\{.*\}", text, flags=re.S)
        if not match:
            raise ValueError("No JSON object found")
        obj = json.loads(match.group(0))
        if not isinstance(obj, dict):
            raise ValueError("Extracted value is not a JSON object")
        return obj

def coerce_allowed(value: Any, allowed: List[str], default: str) -> str:
    text = clean_scalar(value).lower().replace("-", "_").replace(" ", "_")
    return text if text in allowed else default

def coerce_bool(value: Any, default: bool = False) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        if value.strip().lower() in {"true", "1", "yes", "y"}:
            return True
        if value.strip().lower() in {"false", "0", "no", "n"}:
            return False
    if value is None:
        return default
    return bool(value)

def coerce_reference_list(value: Any) -> List[str]:
    if isinstance(value, list):
        raw_items = value
    elif isinstance(value, str):
        text = value.strip()
        if not text:
            raw_items = []
        else:
            raw_items = re.split(r"\s*\|\s*|\s*;\s*|\s*,\s*", text)
    else:
        raw_items = []

    result = []
    seen = set()
    for item in raw_items:
        cleaned = clean_scalar(item)
        if cleaned and cleaned not in seen:
            seen.add(cleaned)
            result.append(cleaned)
    return result

def normalize_llm_payload(payload: Dict[str, Any], row: pd.Series) -> Dict[str, Any]:
    status = coerce_allowed(
        payload.get("validation_status"),
        ALLOWED_VALIDATION_STATUS,
        "unknown_unclear",
    )
    reference_type = coerce_allowed(
        payload.get("reference_type"),
        ["commission_case_number", "commission_decision_reference", "none", "unknown"],
        "unknown",
    )
    confidence = coerce_allowed(
        payload.get("confidence"),
        ALLOWED_CONFIDENCE,
        "low",
    )
    is_valid = coerce_bool(
        payload.get("is_valid_commission_reference"),
        default=status in {"valid_commission_reference", "malformed_but_recoverable"},
    )

    cleaned_reference = clean_scalar(payload.get("cleaned_reference"))
    references = coerce_reference_list(payload.get("all_commission_references"))

    if cleaned_reference and cleaned_reference not in references:
        references.insert(0, cleaned_reference)

    if status == "false_positive":
        is_valid = False
        reference_type = "none"
        cleaned_reference = ""
        references = []
    elif status in {"valid_commission_reference", "malformed_but_recoverable"}:
        is_valid = True
        expected_type = (
            "commission_case_number"
            if clean_scalar(row.get("source_dataset")) == "caseno"
            else "commission_decision_reference"
        )
        if reference_type not in {"commission_case_number", "commission_decision_reference"}:
            reference_type = expected_type
        if not cleaned_reference:
            cleaned_reference = clean_scalar(row.get("normalized_candidate"))
        if not cleaned_reference:
            cleaned_reference = clean_scalar(row.get("normalized_case_number"))
        if cleaned_reference and cleaned_reference not in references:
            references.insert(0, cleaned_reference)
    else:
        is_valid = False
        if status == "unknown_unclear":
            reference_type = "unknown"

    return {
        "is_valid_commission_reference": is_valid,
        "reference_type": reference_type,
        "validation_status": status,
        "cleaned_reference": cleaned_reference,
        "all_commission_references": " | ".join(references),
        "n_commission_references_extracted": len(references),
        "additional_commission_case_text": clean_scalar(payload.get("additional_commission_case_text")),
        "evidence": clean_scalar(payload.get("evidence")),
        "normalization_notes": clean_scalar(payload.get("normalization_notes")),
        "confidence": confidence,
        "reason": clean_scalar(payload.get("reason")),
    }

def _chat_body(prompt: str, system_prompt: str) -> Dict[str, Any]:
    return {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
    }

def call_llm(prompt: str, system_prompt: str) -> str:
    if not REQUESTS_AVAILABLE:
        raise RuntimeError("requests is not available")
    response = requests.post(API_URL, json=_chat_body(prompt, system_prompt), timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]

async def async_call_llm(session: "aiohttp.ClientSession", prompt: str, system_prompt: str) -> str:
    if not AIOHTTP_AVAILABLE:
        raise RuntimeError("aiohttp is not available")
    async with session.post(API_URL, json=_chat_body(prompt, system_prompt)) as response:
        response.raise_for_status()
        data = await response.json()
        return data["choices"][0]["message"]["content"]

def result_base_for_row(row: pd.Series) -> Dict[str, Any]:
    return {
        "llm_case_id": clean_scalar(row.get("llm_case_id")),
        "document_case_key": clean_scalar(row.get("document_case_key")),
        "document_id": clean_scalar(row.get("document_id")),
        "case_system_precheck": clean_scalar(row.get("case_system")),
        "normalized_case_number": clean_scalar(row.get("normalized_case_number")),
        "matched_texts": clean_scalar(row.get("matched_texts")),
        "source_dataset": clean_scalar(row.get("source_dataset")),
        "source_input_file": clean_scalar(row.get("source_input_file")),
        "source_type": clean_scalar(row.get("source_type")),
    }

def error_result_for_row(row: pd.Series, exc: Exception | str, raw_response: str = "", run_id: str = "error") -> Dict[str, Any]:
    return {
        **result_base_for_row(row),
        "is_valid_commission_reference": False,
        "reference_type": "unknown",
        "validation_status": "unknown_unclear",
        "cleaned_reference": "",
        "all_commission_references": "",
        "n_commission_references_extracted": 0,
        "additional_commission_case_text": "",
        "evidence": "",
        "normalization_notes": "",
        "confidence": "low",
        "reason": "LLM call or JSON parsing failed.",
        "llm_status": "error",
        "llm_error": repr(exc) if not isinstance(exc, str) else exc,
        "model": MODEL,
        "classified_at": datetime.now().isoformat(timespec="seconds"),
        "batch_id": run_id,
        "raw_response": raw_response,
    }

def ok_result_from_raw(row: pd.Series, raw: str, run_id: str) -> Dict[str, Any]:
    parsed = extract_json_object(raw)
    normalized = normalize_llm_payload(parsed, row)
    return {
        **result_base_for_row(row),
        **normalized,
        "llm_status": "ok",
        "llm_error": "",
        "model": MODEL,
        "classified_at": datetime.now().isoformat(timespec="seconds"),
        "batch_id": run_id,
        "raw_response": raw,
    }

def classify_row(row: pd.Series, run_id: str = "sequential") -> Dict[str, Any]:
    raw = call_llm(build_prompt(row), get_system_prompt(row))
    return ok_result_from_raw(row, raw, run_id)

async def classify_row_async(
    session: "aiohttp.ClientSession",
    row: pd.Series,
    semaphore: asyncio.Semaphore,
) -> Dict[str, Any]:
    async with semaphore:
        raw = ""
        try:
            raw = await async_call_llm(session, build_prompt(row), get_system_prompt(row))
            return ok_result_from_raw(row, raw, f"async_{CONCURRENT_REQUESTS}")
        except Exception as exc:
            return error_result_for_row(row, exc, raw, f"async_error_{CONCURRENT_REQUESTS}")


## 5. JSONL checkpoint and resume logic

In [6]:
def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                continue
    return rows

def append_jsonl(path: Path, row: Dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")

def latest_by_id(rows: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    latest = {}
    for row in rows:
        key = clean_scalar(row.get("llm_case_id"))
        if key:
            latest[key] = row
    return latest

if RUN_MODE == "redo_all":
    raise RuntimeError(
        "v5 is configured to protect the completed v4 checkpoint. "
        "Create a separate v5 checkpoint path before using redo_all."
    )

if RUN_MODE == "update":
    raise RuntimeError(
        "v5 defaults to rebuild-only and protects the completed v4 checkpoint. "
        "Create a separate writable checkpoint before using update."
    )

checkpoint_rows = read_jsonl(CHECKPOINT_JSONL_PATH)
existing_map = latest_by_id(checkpoint_rows)

checkpoint_line_count = len(checkpoint_rows)
checkpoint_unique_count = len(existing_map)
checkpoint_duplicate_count = checkpoint_line_count - checkpoint_unique_count

print("JSONL lines read:", f"{checkpoint_line_count:,}")
print("Unique llm_case_id values:", f"{checkpoint_unique_count:,}")
print(
    "Superseded/repeated checkpoint lines:",
    f"{checkpoint_duplicate_count:,}",
)

if checkpoint_unique_count == 0:
    raise ValueError(
        "The InfoCuria checkpoint contains no usable llm_case_id records."
    )

checkpoint_status = pd.Series(
    [
        clean_scalar(row.get("llm_status")) or "<blank>"
        for row in existing_map.values()
    ]
).value_counts(dropna=False)
print("Latest checkpoint status distribution:")
print(checkpoint_status.to_string())

if RUN_MODE == "rebuild_excel":
    rows_to_classify = work_cases.iloc[0:0].copy()
elif RUN_MODE == "update":
    rows_to_classify = work_cases[
        ~work_cases["llm_case_id"].isin(existing_map.keys())
    ].copy()
else:
    rows_to_classify = work_cases.copy()

rows_to_classify_caseno = rows_to_classify[
    rows_to_classify["source_dataset"].eq("caseno")
].copy()
rows_to_classify_reference = rows_to_classify[
    rows_to_classify["source_dataset"].eq("reference")
].copy()

print("Rows to classify now:", len(rows_to_classify))
print("  Case-number pass:", len(rows_to_classify_caseno))
print("  Decision-reference pass:", len(rows_to_classify_reference))

for pass_name, pass_rows in [
    ("CASE NUMBER", rows_to_classify_caseno),
    ("DECISION REFERENCE", rows_to_classify_reference),
]:
    if len(pass_rows):
        sample_row = pass_rows.iloc[0]
        sample_prompt = build_prompt(sample_row)
        print(f"\n{pass_name} system prompt length:", len(get_system_prompt(sample_row)))
        print(f"{pass_name} user prompt length:", len(sample_prompt))
        print(sample_prompt[:3000])


JSONL lines read: 14,991
Unique llm_case_id values: 14,983
Superseded/repeated checkpoint lines: 8
Latest checkpoint status distribution:
ok       14978
error        5
Rows to classify now: 0
  Case-number pass: 0
  Decision-reference pass: 0


## 6. Run classification

This keeps the template notebook's native Jupyter asynchronous pattern. Each row is one independent request; concurrency 8 allows vLLM to continuous-batch requests internally.

Set `DRY_RUN = True` in the configuration cell to inspect row selection and prompts without calling the model.


In [7]:
async def run_async_pipeline(rows_df: pd.DataFrame, pass_label: str) -> List[Dict[str, Any]]:
    if not AIOHTTP_AVAILABLE:
        raise RuntimeError("aiohttp is not available. Install it with: pip install aiohttp")

    if rows_df.empty:
        return []

    semaphore = asyncio.Semaphore(int(CONCURRENT_REQUESTS))
    timeout = aiohttp.ClientTimeout(total=REQUEST_TIMEOUT)
    connector = aiohttp.TCPConnector(
        limit=max(int(CONCURRENT_REQUESTS) * 2, 16),
        limit_per_host=max(int(CONCURRENT_REQUESTS) * 2, 16),
        ttl_dns_cache=300,
        enable_cleanup_closed=True,
    )

    results = []
    tasks = []

    try:
        async with aiohttp.ClientSession(
            connector=connector,
            timeout=timeout,
            raise_for_status=False,
        ) as session:
            tasks = [
                asyncio.create_task(classify_row_async(session, row, semaphore))
                for _, row in rows_df.iterrows()
            ]

            completed = asyncio.as_completed(tasks)
            if USE_ASYNC_PROGRESS_BAR:
                completed = tqdm(
                    completed,
                    total=len(tasks),
                    desc=f"{pass_label} async x{CONCURRENT_REQUESTS}",
                    unit="row",
                )

            for task in completed:
                result = await task
                results.append(result)
                append_jsonl(CHECKPOINT_JSONL_PATH, result)
    finally:
        for task in tasks:
            if not task.done():
                task.cancel()
        if tasks:
            await asyncio.gather(*tasks, return_exceptions=True)
        await asyncio.sleep(0.25)

    return results


def retry_error_rows_sequentially(
    original_rows: pd.DataFrame,
    async_results: List[Dict[str, Any]],
    pass_label: str,
) -> List[Dict[str, Any]]:
    if not RETRY_ERRORS_SEQUENTIALLY:
        return []

    failed_ids = [
        clean_scalar(result.get("llm_case_id"))
        for result in async_results
        if clean_scalar(result.get("llm_status")) == "error"
    ]
    failed_ids = [x for x in failed_ids if x]
    if RETRY_MAX_ROWS is not None:
        failed_ids = failed_ids[:int(RETRY_MAX_ROWS)]

    if not failed_ids:
        return []

    lookup = {
        clean_scalar(row.get("llm_case_id")): row
        for _, row in original_rows.iterrows()
    }

    retry_results = []
    for llm_case_id in tqdm(
        failed_ids,
        desc=f"{pass_label} retries",
        unit="row",
    ):
        row = lookup.get(llm_case_id)
        if row is None:
            continue
        raw = ""
        try:
            result = classify_row(row, run_id="sequential_retry")
        except Exception as exc:
            result = error_result_for_row(row, exc, raw, "sequential_retry_error")
        retry_results.append(result)
        append_jsonl(CHECKPOINT_JSONL_PATH, result)

    return retry_results


async def run_validation_pass(rows_df: pd.DataFrame, pass_label: str) -> Dict[str, int]:
    if rows_df.empty:
        print(f"{pass_label}: no rows require classification.")
        return {"rows": 0, "results": 0, "retries": 0}

    print(f"\\nStarting {pass_label}: {len(rows_df):,} rows")
    pass_started = time.time()

    if ASYNC_CONCURRENT_REQUESTS:
        async_results = await run_async_pipeline(rows_df, pass_label)
        retry_results = retry_error_rows_sequentially(
            rows_df,
            async_results,
            pass_label,
        )
        result_count = len(async_results)
        retry_count = len(retry_results)
    else:
        result_count = 0
        retry_count = 0
        for _, row in tqdm(
            rows_df.iterrows(),
            total=len(rows_df),
            desc=f"{pass_label} sequential",
            unit="row",
        ):
            try:
                result = classify_row(row)
            except Exception as exc:
                result = error_result_for_row(row, exc)
            append_jsonl(CHECKPOINT_JSONL_PATH, result)
            result_count += 1

    elapsed = time.time() - pass_started
    print(
        f"Finished {pass_label} in {elapsed:,.1f} seconds | "
        f"results={result_count:,} | retries={retry_count:,}"
    )
    return {
        "rows": len(rows_df),
        "results": result_count,
        "retries": retry_count,
    }


if DRY_RUN:
    print("DRY_RUN=True: no LLM requests sent.")
elif RUN_MODE == "rebuild_excel":
    print("RUN_MODE=rebuild_excel: skipping LLM calls.")
elif rows_to_classify.empty:
    print("No rows require classification.")
else:
    started = time.time()

    # Two explicit pass-throughs, each with its own prompt family and tqdm bar.
    caseno_stats = await run_validation_pass(
        rows_to_classify_caseno,
        "Pass 1/2 — Commission case numbers",
    )
    reference_stats = await run_validation_pass(
        rows_to_classify_reference,
        "Pass 2/2 — Commission decision references",
    )

    print(f"\\nAll validation passes completed in {time.time() - started:,.1f} seconds.")
    print("Case-number pass:", caseno_stats)
    print("Decision-reference pass:", reference_stats)


RUN_MODE=rebuild_excel: skipping LLM calls.


## 7. Rebuild v5 Excel outputs from the completed v4 JSONL checkpoint

This section merges the latest JSONL result for each `llm_case_id` back onto
the current deduplicated InfoCuria regex inputs.

With `RUN_MODE = "rebuild_excel"`:

- no LLM request is made;
- the v4 JSONL is read only;
- repeated JSONL lines are resolved by keeping the latest result per ID;
- new v5 Excel outputs are written.


In [8]:
latest_results = latest_by_id(read_jsonl(CHECKPOINT_JSONL_PATH))
result_df = pd.DataFrame(latest_results.values())

if result_df.empty:
    enriched = cases.copy()

    for col, default in {
        "is_valid_commission_reference": False,
        "reference_type": "",
        "validation_status": "",
        "cleaned_reference": "",
        "all_commission_references": "",
        "n_commission_references_extracted": 0,
        "additional_commission_case_text": "",
        "evidence": "",
        "normalization_notes": "",
        "confidence": "",
        "reason": "",
        "llm_status": "",
        "llm_error": "",
        "model": "",
        "classified_at": "",
        "batch_id": "",
        "raw_response": "",
    }.items():
        enriched[col] = default

else:
    # Remove accidental duplicate column labels.
    result_df = result_df.loc[:, ~result_df.columns.duplicated()].copy()

    if "llm_case_id" not in result_df.columns:
        raise ValueError(
            "The rebuilt JSONL results do not contain 'llm_case_id'. "
            "Check the JSONL checkpoint contents."
        )

    # Keep the merge key only once.
    result_columns = [
        c
        for c in result_df.columns
        if c not in {
            "llm_case_id",
            "document_case_key",
            "document_id",
            "case_system_precheck",
            "normalized_case_number",
            "matched_texts",
            "source_type",
            "source_dataset",
            "source_input_file",
        }
    ]

    result_for_merge = result_df[
        ["llm_case_id", *result_columns]
    ].copy()

    # Defensive safeguard in case the JSONL contains repeated results
    # for the same case ID. Keep the latest result.
    if result_for_merge["llm_case_id"].duplicated().any():
        duplicate_count = int(
            result_for_merge["llm_case_id"].duplicated().sum()
        )

        print(
            f"Warning: found {duplicate_count:,} duplicate llm_case_id "
            "rows in the rebuilt results. Keeping the latest result."
        )

        result_for_merge = result_for_merge.drop_duplicates(
            subset=["llm_case_id"],
            keep="last",
        )

    if cases["llm_case_id"].duplicated().any():
        duplicate_count = int(cases["llm_case_id"].duplicated().sum())

        raise ValueError(
            f"The input cases contain {duplicate_count:,} duplicate "
            "llm_case_id values. The deduplicated input view should contain "
            "one row per llm_case_id."
        )

    enriched = cases.merge(
        result_for_merge,
        on="llm_case_id",
        how="left",
        validate="one_to_one",
    )

front = [
    c
    for c in [
        "llm_case_id",
        "source_dataset",
        "source_input_file",
        "source_type",
        "document_case_key",
        "document_identifier_key",
        "document_id",
        "infocuria_document_id",
        "source_row_i",
        "source_entry_id",
        "case_number",
        "case_numbers",
        "document_type",
        "document_date",
        "decision_date",
        "language",
        "document_title",
        "title",
        "case_system",
        "reference_category",
        "commission_reference_type",
        "normalized_case_number",
        "normalized_candidate",
        "matched_texts",
        "hit_types",
        "commission_units",
        "commission_reference_types",
        "decision_suffixes",
        "derived_celex_values",
        "n_hits",
        "is_valid_commission_reference",
        "reference_type",
        "validation_status",
        "cleaned_reference",
        "all_commission_references",
        "n_commission_references_extracted",
        "additional_commission_case_text",
        "evidence",
        "normalization_notes",
        "confidence",
        "reason",
        "llm_status",
        "llm_error",
        "model",
        "classified_at",
        "batch_id",
        "paragraph_text_agg",
        "context_180_agg",
        "context_agg",
        "downloaded_file_path",
        "resolved_text_path",
        "quality_flag",
        "quality_reason",
        "cleaning_success",
        "clean_success",
        "cleaning_status",
        "download_status",
        "extraction_method",
        "extraction_engine",
        "mineru_attempted",
        "mineru_used",
        "raw_response",
    ]
    if c in enriched.columns
]

other = [
    c
    for c in enriched.columns
    if c not in front
]

enriched = enriched[front + other]

# Rebuild integrity checks.
matched_results = int(enriched["llm_status"].notna().sum()) if "llm_status" in enriched.columns else 0
missing_results = len(enriched) - matched_results

print("Rebuild integrity:")
print("  Input rows:", f"{len(enriched):,}")
print("  Rows matched to checkpoint results:", f"{matched_results:,}")
print("  Rows without checkpoint results:", f"{missing_results:,}")

if missing_results:
    print(
        "WARNING: some current input rows have no matching result in the "
        "completed v4 checkpoint. They remain in the Excel output with blank "
        "validation fields; no LLM calls are made in rebuild mode."
    )

# Optional combined audit output, plus the two requested validated workbooks.
enriched.to_excel(COMBINED_CLASSIFIED_EXCEL_PATH, index=False)

caseno_validated = enriched[enriched["source_dataset"].eq("caseno")].copy()
reference_validated = enriched[enriched["source_dataset"].eq("reference")].copy()

caseno_validated.to_excel(output_caseno_path, index=False)
reference_validated.to_excel(output_ref_path, index=False)

print(f"Wrote {len(caseno_validated):,} case-number rows to {output_caseno_path}")
print(f"Wrote {len(reference_validated):,} decision-reference rows to {output_ref_path}")
print(f"Wrote combined audit Excel to {COMBINED_CLASSIFIED_EXCEL_PATH}")

enriched.head(5)

Rebuild integrity:
  Input rows: 14,983
  Rows matched to checkpoint results: 14,983
  Rows without checkpoint results: 0
Wrote 4,936 case-number rows to /home/edik/projects/eccjeu/output/court_infocuria/excel/court_infocuria_commission_caseno_hit_validated_v5.xlsx
Wrote 10,047 decision-reference rows to /home/edik/projects/eccjeu/output/court_infocuria/excel/court_infocuria_commission_reference_hit_validated_v5.xlsx
Wrote combined audit Excel to /home/edik/projects/eccjeu/output/court_infocuria/excel/court_infocuria_commission_two_prompts_validated_v5_combined.xlsx


,llm_case_id,source_dataset,source_input_file,source_type,document_case_key,document_identifier_key,document_id,infocuria_document_id,source_row_i,source_entry_id,...,first_hit_end,all_hit_starts,needs_llm_validation,confidence_precheck,file_type,file_format,raw_file_path,file_name,resolved_text_path_source,normalized_commission_decision_reference
0,caseno::1055241||commission_case_number||COMP/...,caseno,court_infocuria_commission_caseno_hit_deduplic...,court_infocuria_commission_caseno_hit,1055241||commission_case_number||COMP/C2/38.698,1055241||commission_case_number||COMP/C2/38.698,1055241,1055241,10074,10074,...,1457,1443,True,high,html,html,/home/edik/projects/eccjeu/data/raw/court_info...,85042-NL-1.html,clean_text_path,NaN
1,caseno::1204995||commission_case_number||COMP/...,caseno,court_infocuria_commission_caseno_hit_deduplic...,court_infocuria_commission_caseno_hit,1204995||commission_case_number||COMP/E-1/36.212,1204995||commission_case_number||COMP/E-1/36.212,1204995,1204995,26140,26140,...,2795,2780 | 75080,True,high,html,html,/home/edik/projects/eccjeu/data/raw/court_info...,73086-EN-1.html,clean_text_path,NaN
2,caseno::1211704||commission_case_number||COMP/...,caseno,court_infocuria_commission_caseno_hit_deduplic...,court_infocuria_commission_caseno_hit,1211704||commission_case_number||COMP/38.543,1211704||commission_case_number||COMP/38.543,1211704,1211704,54452,54452,...,797,786 | 1402,True,high,html,html,/home/edik/projects/eccjeu/data/raw/court_info...,85097-EN-1.html,clean_text_path,NaN
3,caseno::1211705||commission_case_number||COMP/...,caseno,court_infocuria_commission_caseno_hit_deduplic...,court_infocuria_commission_caseno_hit,1211705||commission_case_number||COMP/F/38.620,1211705||commission_case_number||COMP/F/38.620,1211705,1211705,53685,53685,...,1083,1070 | 3729,True,high,html,html,/home/edik/projects/eccjeu/data/raw/court_info...,85098-EN-1.html,clean_text_path,NaN
4,caseno::1211705||commission_case_number||COMP/...,caseno,court_infocuria_commission_caseno_hit_deduplic...,court_infocuria_commission_caseno_hit,1211705||commission_case_number||COMP/F/38.645,1211705||commission_case_number||COMP/F/38.645,1211705,1211705,53685,53685,...,160043,160030,True,high,html,html,/home/edik/projects/eccjeu/data/raw/court_info...,85098-EN-1.html,clean_text_path,NaN


## 8. Document-level summary and rerun candidates

In [9]:
def join_unique(values: Iterable[Any], max_items: int = 30) -> str:
    result = []
    seen = set()
    for value in values:
        for item in clean_scalar(value).split(" | "):
            item = clean_scalar(item)
            if item and item not in seen:
                seen.add(item)
                result.append(item)
            if len(result) >= max_items:
                return " | ".join(result)
    return " | ".join(result)

tmp = enriched.copy()
tmp["is_valid_bool"] = (
    tmp["is_valid_commission_reference"].fillna(False).astype(bool)
    if "is_valid_commission_reference" in tmp.columns
    else False
)
tmp["n_extracted_num"] = pd.to_numeric(
    tmp.get("n_commission_references_extracted", 0),
    errors="coerce",
).fillna(0).astype(int)

if tmp.empty:
    document_summary = pd.DataFrame()
else:
    group = tmp.groupby(["source_dataset", "document_id"], dropna=False)
    document_summary = group.agg(
        n_commission_candidate_rows=("llm_case_id", "count"),
        n_regex_hits_aggregated=("n_hits", lambda s: int(pd.to_numeric(s, errors="coerce").fillna(0).sum())),
        n_llm_ok=("llm_status", lambda s: int(s.fillna("").eq("ok").sum())),
        n_valid_commission_references=("is_valid_bool", "sum"),
        n_valid_clean_structure=("validation_status", lambda s: int(s.fillna("").eq("valid_commission_reference").sum())),
        n_valid_case_numbers=("reference_type", lambda s: int(s.fillna("").eq("commission_case_number").sum())),
        n_valid_decision_references=("reference_type", lambda s: int(s.fillna("").eq("commission_decision_reference").sum())),
        n_malformed_but_recoverable=("validation_status", lambda s: int(s.fillna("").eq("malformed_but_recoverable").sum())),
        n_false_positives=("validation_status", lambda s: int(s.fillna("").eq("false_positive").sum())),
        n_unknown_unclear=("validation_status", lambda s: int(s.fillna("").eq("unknown_unclear").sum())),
        n_commission_references_extracted=("n_extracted_num", "sum"),
        validated_commission_references=("all_commission_references", join_unique),
        original_regex_candidates=("normalized_case_number", join_unique),
        commission_case_text_examples=("additional_commission_case_text", lambda s: join_unique(s, max_items=10)),
        confidence_levels=("confidence", join_unique),
    ).reset_index()

    metadata_columns = [
        c for c in [
            "source_dataset",
            "document_id",
            "infocuria_document_id",
            "source_row_i",
            "source_entry_id",
            "case_number",
            "case_numbers",
            "document_title",
            "title",
            "document_type",
            "document_date",
            "decision_date",
            "language",
            "downloaded_file_path",
            "file_type",
            "file_name",
            "file_format",
            "raw_file_path",
            "quality_flag",
            "quality_reason",
            "cleaning_success",
            "clean_success",
            "cleaning_status",
            "download_status",
            "extraction_method",
            "extraction_engine",
            "mineru_attempted",
            "mineru_used",
            "resolved_text_path",
            "resolved_text_path_source",
        ] if c in tmp.columns
    ]
    metadata = tmp[metadata_columns].drop_duplicates(["source_dataset", "document_id"])
    document_summary = metadata.merge(
        document_summary,
        on=["source_dataset", "document_id"],
        how="right",
    )

document_summary.to_excel(DOCUMENT_SUMMARY_PATH, index=False)

rerun_candidates = enriched[
    enriched.get("llm_status", "").fillna("").eq("error")
    | enriched.get("confidence", "").fillna("").eq("low")
    | enriched.get("validation_status", "").fillna("").eq("unknown_unclear")
].copy()

rerun_candidates.to_excel(RERUN_CANDIDATES_PATH, index=False)

print("Document summary:", DOCUMENT_SUMMARY_PATH, "| rows:", len(document_summary))
print("Rerun candidates:", RERUN_CANDIDATES_PATH, "| rows:", len(rerun_candidates))


Document summary: /home/edik/projects/eccjeu/output/court_infocuria/excel/court_infocuria_commission_two_prompts_validated_v5_document_summary.xlsx | rows: 8545
Rerun candidates: /home/edik/projects/eccjeu/output/court_infocuria/excel/court_infocuria_commission_two_prompts_validated_v5_rerun_candidates.xlsx | rows: 5


## 9. v5 outputs and interpretation

The notebook writes:

- `court_infocuria_commission_caseno_hit_validated_v5.xlsx`
- `court_infocuria_commission_reference_hit_validated_v5.xlsx`
- `court_infocuria_commission_two_prompts_validated_v5_combined.xlsx`
- `court_infocuria_commission_two_prompts_validated_v5_document_summary.xlsx`
- `court_infocuria_commission_two_prompts_validated_v5_rerun_candidates.xlsx`

The existing
`court_infocuria_commission_two_prompts_validated_v4.jsonl` remains the
classification source of truth and is not rerun or overwritten.

Interpretation:

- `valid_commission_reference`: genuine Commission reference with a defensible
  normalized identifier.
- `malformed_but_recoverable`: genuine reference repaired from OCR, spacing,
  punctuation, or normalization damage.
- `false_positive`: not a Commission or High Authority decision/case reference.
- `unknown_unclear`: insufficient contextual evidence.
